In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.staronedov.lesson3 import Exercise

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target

X = X / 16.0

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
def evaluate_hyperparameters(batch_size, learning_rate, n_epochs=50):
    """Обучает модель с заданными гиперпараметрами и возвращает точность на валидации."""
    rng = np.random.default_rng(42)
    model = Exercise.create_model(
        Exercise.create_linear_layer(64, 128, rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(128, 10, rng),
        Exercise.create_logsoftmax_layer(),
    )
    loss_fn = Exercise.create_nll_loss()

    n_samples = X_train.shape[0]
    indices = np.arange(n_samples)

    for _epoch in range(n_epochs):
        np.random.shuffle(indices)
        for i in range(0, n_samples, batch_size):
            batch_idx = indices[i : i + batch_size]
            x_batch = X_train[batch_idx]
            y_batch = y_train[batch_idx]

            log_probs = model.forward(x_batch)
            loss_fn.forward(log_probs, y_batch)

            d_loss = loss_fn.backward()
            model.backward(d_loss)
            for param, grad in zip(model.parameters, model.grad, strict=True):
                param -= learning_rate * grad

    val_log_probs = model.forward(X_val)
    val_probs = np.exp(val_log_probs)
    preds = np.argmax(val_probs, axis=1)
    val_acc = np.mean(preds == y_val)
    return val_acc

In [ ]:
batch_sizes = [8, 16, 32, 64, 128]
learning_rates = [0.001, 0.003, 0.01, 0.03, 0.1, 0.2, 0.3]

results = {}
print("\nИсследование гиперпараметров:\n")
print(f"{'Batch Size':>10} | {'Learning Rate':>12} | Validation Accuracy")
print("-" * 45)

for bs in batch_sizes:
    for lr in learning_rates:
        acc = evaluate_hyperparameters(bs, lr, n_epochs=50)
        results[(bs, lr)] = acc
        print(f"{bs:>10} | {lr:>12} | {acc:.4f}")

best_bs, best_lr = max(results, key=lambda k: results[k])
best_acc_val = results[(best_bs, best_lr)]

print("\nЛучшие гиперпараметры:")
print(f"  Batch size: {best_bs}")
print(f"  Learning rate: {best_lr}")
print(f"  Validation accuracy: {best_acc_val:.4f}")

In [ ]:
X_train_val = np.vstack([X_train, X_val])
y_train_val = np.hstack([y_train, y_val])


rng = np.random.default_rng(42)
final_model = Exercise.create_model(
    Exercise.create_linear_layer(64, 128, rng),
    Exercise.create_relu_layer(),
    Exercise.create_linear_layer(128, 10, rng),
    Exercise.create_logsoftmax_layer(),
)
loss_fn = Exercise.create_nll_loss()

n_samples = X_train_val.shape[0]
indices = np.arange(n_samples)
n_epochs_final = 50

for _epoch in range(n_epochs_final):
    np.random.shuffle(indices)
    for i in range(0, n_samples, best_bs):
        batch_idx = indices[i : i + best_bs]
        x_batch = X_train_val[batch_idx]
        y_batch = y_train_val[batch_idx]

        log_probs = final_model.forward(x_batch)
        loss = loss_fn.forward(log_probs, y_batch)

        d_loss = loss_fn.backward()
        final_model.backward(d_loss)
        for param, grad in zip(final_model.parameters, final_model.grad, strict=True):
            param -= best_lr * grad

In [ ]:
test_log_probs = final_model.forward(X_test)
test_probs = np.exp(test_log_probs)
preds = np.argmax(test_probs, axis=1)
test_acc = np.mean(preds == y_test)

print("\nФинальная проверка на тесте:")
print(f"  Test accuracy: {test_acc:.4f}")
print(f"  Разница (val - test): {best_acc_val - test_acc:.4f}")